# 03 - Hybrid quantum-classical model (Model B)
Classical feature layer -> 4-qubit PQC (Ry encoding, trainable Ry/Rz, CNOT chain, final Ry) -> <Z_i> -> classical output layer.

Full multi-seed experiments live in `experiments/`; this notebook runs one seed end-to-end.

In [ ]:
# --- Colab setup: run once, then (if prompted) restart the runtime ---
# Replace the URL with your own GitHub repo after you push this project.
import os
if not os.path.exists("src"):
    !git clone https://github.com/<your-username>/hybrid-quantum-classifier.git
    %cd hybrid-quantum-classifier
%pip install -q -r requirements.txt

In [ ]:
from src.quantum_circuit import build_circuit
qc, x, w = build_circuit(n_qubits=4, n_layers=1, final_rotation=True)
print("input params:", len(x), "| trainable circuit params:", len(w))
qc.draw("mpl", fold=-1)

## Gradient check (do this BEFORE training)
A circuit that runs is not necessarily one that trains. We check that every circuit parameter has a non-zero gradient. This check is what revealed that the textbook ansatz (without the final `Ry` layer) has *dead* `Rz` parameters.

In [ ]:
import numpy as np
from src.quantum_circuit import build_qnn, n_weights

def weight_grad_magnitudes(final_rotation):
    rng = np.random.default_rng(0)
    qnn = build_qnn(4, 1, final_rotation, precision=0.0)
    xs = rng.uniform(-2.5, 2.5, (6, 4)); ws = rng.uniform(-3, 3, n_weights(4, 1, final_rotation))
    _, wg = qnn.backward(xs, ws)
    return np.abs(wg).sum(axis=(0, 1))

print("with final Ry layer :", weight_grad_magnitudes(True).round(3))
g = weight_grad_magnitudes(False)
print("literal ansatz      :", g.round(3), "<- Rz entries (odd indices) are exactly 0")

In [ ]:
import torch
from src.data_preprocessing import load_binary_iris
from src.hybrid_model import HybridQNN, count_params
from src.train import set_seed, train_model
from src.evaluate import evaluate

TASK, SEED, EPOCHS = "versicolor_vs_virginica", 0, 20      # ~3 min on a free Colab CPU
set_seed(SEED)
data = load_binary_iris(TASK, seed=SEED)
model = HybridQNN(n_qubits=4, n_layers=1, final_rotation=True, seed=SEED)
print("trainable parameters:", count_params(model))
w_before = model.qnn.weight.detach().clone()
hist = train_model(model, data, epochs=EPOCHS, lr=0.05, batch_size=16, seed=SEED, verbose=True)
print("quantum weights changed during training:", not torch.equal(w_before, model.qnn.weight.detach()))
m = evaluate(model, data.X_test, data.y_test)
print({k: (round(v, 3) if not isinstance(v, list) else v) for k, v in m.items()})
print("training time: %.1fs" % hist["train_time_s"])

In [ ]:
import matplotlib.pyplot as plt
plt.plot(hist["train_loss"], label="train loss"); plt.plot(hist["test_loss"], label="test loss (monitoring only)")
plt.xlabel("epoch"); plt.ylabel("BCE"); plt.legend(); plt.title("Hybrid model"); plt.show()

## Reproduce all experiments
```bash
python -m experiments.run_all      # ~1 h on one CPU core, resumable
python -m experiments.make_report  # tables + figures into results/
```